# AETHER STT — CTC-upsampler fix: smoke test

Validates the CTC-upsampler fix before committing to a full run.

**What was wrong:** at Mimi's raw 12.5Hz semantic frame rate, byte-level CTC
needs at least one input frame per target byte (plus separators for
adjacent repeats). Measured on the actual baseline train cache: only
**15.7%** of examples (20757 / 132553) survived the CTC-feasibility filter,
and even those sat at a mean required/available-frame ratio of ~0.89
(p95/p99 basically at the 1.0 ceiling — no slack for CTC blanks). The
200k-step baseline run's plateau (WER ~77%, hypotheses systematically
shorter than reference, growing train/eval loss gap) all trace back to
this: the model trained on a small, short-utterance-biased ~16% slice of
the intended data.

**The fix** (`src/aether_v3/models/ctc_upsampler.py`): a learned ×4
temporal upsampler between `AetherSpeechEncoder`'s 12.5Hz states and the
CTC head. `AetherSpeechEncoder`'s own 12.5Hz output is untouched (still
feeds the future phase-2 Qwen bridge) - only the CTC branch runs at the
higher effective rate. The extraction-time feasibility filter now checks
against that upsampled rate too, so re-extracting should recover nearly
all of the previously-dropped 84.3%.

This notebook: **(1)** wipes the stale pre-fix cache (its fingerprint
already changed, so `prepare_cache` would refuse to silently reuse it
anyway) and re-extracts with the fixed filter, reporting the new
kept-example count against the 20757/132553 baseline; **(2)** runs a short
(`configs/ctc_upsampler_smoke.yaml`, 8000 steps) training smoke test on the
fixed cache so the fix can be judged on real WER/CER before committing to
a full run.

Meant to be run from VSCode against a GPU kernel - re-run any cell any
time, everything here is idempotent.

In [8]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/karl4th/aether-v3.git"
REPO_NAME = "aether-v3"

cwd = Path.cwd()
if cwd.name == REPO_NAME and (cwd / ".git").is_dir():
    subprocess.run(["git", "pull"], check=True, cwd=cwd)
    repo_dir = cwd
    print("Already inside the repo, pulled.")
else:
    repo_dir = cwd / REPO_NAME
    if (repo_dir / ".git").is_dir():
        subprocess.run(["git", "-C", str(repo_dir), "pull"], check=True)
        print("Pulled")
    elif repo_dir.exists():
        raise RuntimeError(
            f"{repo_dir} exists but isn't a git checkout (no .git/) - "
            "remove or rename it manually before re-running this cell."
        )
    else:
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
        print("Cloned")
    os.chdir(repo_dir)

print("cwd:", os.getcwd())
commit_hash = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=repo_dir, capture_output=True, text=True, check=True
).stdout.strip()
commit_subject = subprocess.run(
    ["git", "log", "-1", "--format=%s"], cwd=repo_dir, capture_output=True, text=True, check=True
).stdout.strip()
print("HEAD commit:", commit_hash)
print("HEAD subject:", commit_subject)

Already inside the repo, pulled.
cwd: /content/aether-v3
HEAD commit: e1d570a51d82372e925d49f3f8333713a376fa32
HEAD subject: Implement CTC Upsampler to enhance frame rate for CTC head


In [2]:
import importlib.util
import subprocess
import sys


def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


# Most GPU notebook images already ship a CUDA-matched torch build — don't
# clobber it. Only install if genuinely missing.
if importlib.util.find_spec("torch") is None:
    pip_install("torch")
if importlib.util.find_spec("torchaudio") is None:
    pip_install("torchaudio")

# Union of prepare_data.ipynb's (extraction) and train_ctc.ipynb's
# (training) requirements - this notebook does both.
pip_install(
    "transformers>=5.17",  # <5.17 lacks MimiModel.get_audio_codes_mask - see mimi_wrapper.py
    "datasets>=2.19,<4.0",  # >=4.0 requires torchcodec + system ffmpeg for Audio decoding
    "soundfile",
    "librosa",
    "jiwer",
    "pyyaml",
    "numpy",
    "tqdm",
    "hf_transfer",  # multi-connection downloader - speeds up the raw-audio download
    "huggingface_hub",
)

In [3]:
import logging
import os
import sys

# Must be set before `datasets`/`huggingface_hub` get imported anywhere in
# this kernel - see prepare_data.ipynb for why.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

sys.path.insert(0, os.path.join(os.getcwd(), "src"))

# So prepare_cache's/run_training's own progress logs (extraction %, CTC-
# infeasible %, step/loss/throughput, eval results) actually show up in
# this notebook's output.
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("using device:", DEVICE)
assert DEVICE == "cuda", "No GPU visible on this kernel - check VSCode's connected runtime."


torch: 2.11.0+cu128
CUDA available: True | GPU count: 1
GPU: NVIDIA A100-SXM4-40GB
using device: cuda


## Config + cache location

Same Drive-mount-with-local-fallback pattern as the other notebooks, so
this works whether the kernel VSCode is attached to happens to be Colab
or a plain GPU rental.

In [7]:
from pathlib import Path

from aether_v3.config import load_config

real_config = load_config("configs/ctc_upsampler_smoke.yaml")

try:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/aether-v3")
except ImportError:
    DRIVE_ROOT = Path("drive_cache").resolve()
    print(f"Not running in Colab - falling back to local '{DRIVE_ROOT}' (no cross-session persistence).")

DRIVE_CACHE_DIR = DRIVE_ROOT / "data_cache" / Path(real_config.data.cache_dir).name
DRIVE_RUN_DIR = DRIVE_ROOT / "runs" / Path(real_config.train.output_dir).name
real_config.train.output_dir = str(DRIVE_RUN_DIR)

real_config.data.extraction_num_workers = max(2, min(os.cpu_count() or 4, 16))

print("Drive cache dir:", DRIVE_CACHE_DIR)
print("Drive run dir:  ", DRIVE_RUN_DIR)

TypeError: CTCConfig.__init__() got an unexpected keyword argument 'upsample_factor'

## Step 0 — wipe the stale (pre-fix) cache

The cache fingerprint already changed (`ctc.upsample_factor` is now part
of it), so `prepare_cache` would raise on the old directory rather than
silently reuse it - this just does that cleanup up front, locally and on
the Drive mirror, so Step 1 starts clean. Safe to re-run (no-ops if
already clean).

In [ ]:
import shutil
from pathlib import Path

from aether_v3.data.cache_sync import CACHE_ROLES

for base in (Path(real_config.data.cache_dir), DRIVE_CACHE_DIR):
    for role in CACHE_ROLES:
        role_dir = base / role
        fp_path = base / f"{role}.fingerprint.json"
        if role_dir.exists():
            shutil.rmtree(role_dir)
            print(f"Removed stale cache dir: {role_dir}")
        if fp_path.exists():
            fp_path.unlink()
            print(f"Removed stale fingerprint: {fp_path}")

print("Cache wiped - Step 1 will re-extract from scratch.")

## Hugging Face login (needed for the download)

`openslr/librispeech_asr` / the `distil-whisper` fallback are hitting
anonymous rate limits without this. Set an `HF_TOKEN` env var beforehand
(e.g. via VSCode's env config or `%env HF_TOKEN=...`), or just run this
cell as-is and paste a token when prompted.

In [ ]:
import os

from huggingface_hub import login

token = os.environ.get("HF_TOKEN")
if token:
    login(token=token)
else:
    login()  # prompts for a token interactively

print("Logged in to the Hugging Face Hub.")

## Step 1 — re-download + re-extract with the fixed CTC-feasibility filter

Same two-stage shape as `prepare_data.ipynb` (download fully first, then
extract), collapsed into one cell here since the cache was just wiped
above and there's nothing to "skip re-extraction for" this time. Watch
the `CTC-infeasible` warning line for `train` — it should now report a
much smaller drop than the baseline's 84.3%.

In [ ]:
from aether_v3.data.mimi_cache import download_raw_splits, prepare_cache

download_raw_splits(real_config)
cache_paths = prepare_cache(real_config, device=DEVICE)

from datasets import load_from_disk

print("\nKept-example summary (vs. pre-fix baseline train: 20757/132553 = 15.7%):")
for role, path in cache_paths.items():
    ds = load_from_disk(str(path))
    print(f"  {role:<10} {len(ds)} examples")

In [ ]:
from aether_v3.data.cache_sync import push_to_remote

pushed = push_to_remote(real_config.data.cache_dir, DRIVE_CACHE_DIR)
if pushed:
    print(f"Pushed fixed cache to Drive: {pushed}")

## Step 2 — smoke training run (8000 steps)

Fresh run, separate `output_dir` from the old baseline - nothing here can
collide with or resume from the pre-fix checkpoint.

In [ ]:
from aether_v3.training.train_ctc import run_training

run_training(real_config)

## Monitor

Re-run any time (even while the cell above is still training) to see the
latest curves from `log.jsonl`. Compare against the baseline numbers:
WER started at ~0.95 and plateaued around ~0.77 by step 100k, with eval
loss climbing from a step-6000 minimum of 0.269 to 0.535 by step 106000
while train loss kept falling to ~0.18 - if the fix worked, WER/CER here
should be dropping faster and eval loss should not be diverging like
that.

In [ ]:
import json

import matplotlib.pyplot as plt

with open(f"{real_config.train.output_dir}/log.jsonl") as f:
    rows = [json.loads(line) for line in f]

train_rows = [r for r in rows if "loss" in r and "eval_loss" not in r]
eval_rows = [r for r in rows if "eval_cer" in r]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot([r["step"] for r in train_rows], [r["loss"] for r in train_rows], label="train")
axes[0].plot([r["step"] for r in eval_rows], [r["eval_loss"] for r in eval_rows], label="eval")
axes[0].set_title("loss")
axes[0].set_xlabel("step")
axes[0].legend()

axes[1].plot([r["step"] for r in eval_rows], [r["eval_wer"] for r in eval_rows])
axes[1].set_title("eval WER")
axes[1].set_xlabel("step")

axes[2].plot([r["step"] for r in eval_rows], [r["eval_cer"] for r in eval_rows])
axes[2].set_title("eval CER")
axes[2].set_xlabel("step")
plt.show()

if eval_rows:
    best = min(eval_rows, key=lambda r: r["eval_cer"])
    print("best eval so far:", best)
    latest = eval_rows[-1]
    print("latest eval:", latest)